In [1]:
import sys, numpy as np, plotly
print("Python:", sys.version.split()[0])
print("numpy: ", np.__version__)
print("plotly:", plotly.__version__)


Python: 3.13.9
numpy:  2.4.6
plotly: 6.7.0


In [2]:
import numpy as np

from tbc.synthesis import SimConfig
from tbc.geometry import wrap, min_image_displacement, torus_distance
from tbc.motion import (
    sample_initial_centers,
    sample_initial_velocities,
    step_motion,
    simulate,
    Trajectory,
)
from tbc.collision import resolve_collisions


# -----------------------------
# 1. Create config
# -----------------------------

cfg = SimConfig(
    cube_size=10.0,
    n_spheres=4,
    radius=0.5,
    elasticity=1.0,
    speed=1.0,
    motion_noise_std=0.05,
    n_timesteps=60,
    dt=0.1,
    n_inliers_per_sphere=20,
    n_background=50,
    obs_noise_std=0.1,
    seed=0,
)

rng = np.random.default_rng(cfg.seed)

print("Config OK:")
print(cfg)


# -----------------------------
# 2. Test geometry.py
# -----------------------------

x = np.array([-1.0, 2.0, 11.0])
wrapped = wrap(x, cfg.cube_size)

assert np.all((wrapped >= 0) & (wrapped < cfg.cube_size))

d1 = torus_distance(
    np.array([0.5, 0.0]),
    np.array([9.5, 0.0]),
    cfg.cube_size,
)

d2 = torus_distance(
    np.array([1.0, 1.0]),
    np.array([4.0, 5.0]),
    cfg.cube_size,
)

assert np.isclose(d1, 1.0)
assert np.isclose(d2, 5.0)

print("Geometry tests passed.")


# -----------------------------
# 3. Test initial centers
# -----------------------------

centers = sample_initial_centers(cfg, rng)

assert centers.shape == (cfg.n_spheres, 2)
assert np.all((centers >= 0) & (centers < cfg.cube_size))

for i in range(cfg.n_spheres):
    for j in range(i + 1, cfg.n_spheres):
        d = torus_distance(centers[i], centers[j], cfg.cube_size)
        assert d >= 2 * cfg.radius

print("Initial centers test passed.")


# -----------------------------
# 4. Test initial velocities
# -----------------------------

velocities = sample_initial_velocities(cfg, rng)

assert velocities.shape == (cfg.n_spheres, 2)

speed_norms = np.linalg.norm(velocities, axis=1)
assert np.allclose(speed_norms, cfg.speed)

print("Initial velocities test passed.")


# -----------------------------
# 5. Test one motion step
# -----------------------------

new_centers, new_velocities = step_motion(centers, velocities, cfg, rng)

assert new_centers.shape == (cfg.n_spheres, 2)
assert new_velocities.shape == (cfg.n_spheres, 2)
assert np.all((new_centers >= 0) & (new_centers < cfg.cube_size))

print("Motion step test passed.")


# -----------------------------
# 6. Test collision resolution
# -----------------------------

test_centers = np.array([
    [4.5, 5.0],
    [5.5, 5.0],
])

test_velocities = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
])

collision_cfg = SimConfig(
    cube_size=10.0,
    n_spheres=2,
    radius=0.6,
    elasticity=1.0,
    speed=1.0,
    motion_noise_std=0.0,
    n_timesteps=10,
    dt=0.1,
    n_inliers_per_sphere=20,
    n_background=50,
    obs_noise_std=0.1,
    seed=0,
)

resolved = resolve_collisions(test_centers, test_velocities, collision_cfg)

assert resolved.shape == test_velocities.shape
assert np.allclose(resolved[0], [-1.0, 0.0])
assert np.allclose(resolved[1], [1.0, 0.0])

print("Collision resolution test passed.")


# -----------------------------
# 7. Test full simulation
# -----------------------------

traj = simulate(cfg)

assert isinstance(traj, Trajectory)
assert traj.centers.shape == (cfg.n_timesteps, cfg.n_spheres, 2)
assert traj.velocities.shape == (cfg.n_timesteps, cfg.n_spheres, 2)

assert np.all((traj.centers >= 0) & (traj.centers < cfg.cube_size))
assert not np.isnan(traj.centers).any()
assert not np.isnan(traj.velocities).any()

print("Simulation test passed.")


# -----------------------------
# Final success message
# -----------------------------

print("\nAll Task 2–7 tests passed successfully.")

Config OK:
SimConfig(cube_size=10.0, n_spheres=4, radius=0.5, elasticity=1.0, speed=1.0, motion_noise_std=0.05, n_timesteps=60, dt=0.1, n_inliers_per_sphere=20, n_background=50, obs_noise_std=0.1, seed=0)
Geometry tests passed.
Initial centers test passed.
Initial velocities test passed.
Motion step test passed.
Collision resolution test passed.
Simulation test passed.

All Task 2–7 tests passed successfully.


In [3]:
from tbc.synthesis import generate_dataset, save_dataset, load_dataset
from tbc.motion import simulate

cfg = SimConfig(cube_size=10.0, n_spheres=3, radius=0.5, elasticity=1.0,
                speed=1.0, motion_noise_std=0.05, n_timesteps=10,
                dt=0.1, n_inliers_per_sphere=20, n_background=50,
                obs_noise_std=0.1, seed=0)

ds = generate_dataset(cfg)

T, K, nin, nbg = 10, 3, 20, 50
M_expected = T * (K * nin + nbg)

assert ds.points.shape  == (M_expected, 2),  f"points shape wrong: {ds.points.shape}"
assert ds.times.shape   == (M_expected,),     f"times shape wrong"
assert ds.labels.shape  == (M_expected,),     f"labels shape wrong"
assert (ds.labels == -1).sum() == T * nbg,    f"background count wrong"
assert (ds.labels ==  0).sum() == T * nin,    f"sphere 0 count wrong"

save_dataset(ds, "data/test.npz")
ds2 = load_dataset("data/test.npz")
assert np.array_equal(ds.points, ds2.points), "save/load mismatch"
print("Tasks 8 and 9 passed!")
print(f"Total points M = {ds.points.shape[0]}")
print(f"Background: {(ds.labels == -1).sum()}, Sphere 0: {(ds.labels == 0).sum()}, Sphere 1: {(ds.labels == 1).sum()}")

Tasks 8 and 9 passed!
Total points M = 1100
Background: 500, Sphere 0: 200, Sphere 1: 200


In [4]:
from tbc.synthesis import generate_dataset
from tbc.viz import plot_spacetime, plot_spacetime_dataset, animate_world

cfg_viz = SimConfig(
    cube_size=10.0, n_spheres=4, radius=0.5, elasticity=1.0,
    speed=1.0, motion_noise_std=0.05, n_timesteps=60,
    dt=0.1, n_inliers_per_sphere=20, n_background=50,
    obs_noise_std=0.1, seed=0,
)

ds = generate_dataset(cfg_viz)

plot_spacetime(ds.trajectory, cfg_viz).show()
plot_spacetime_dataset(ds).show()
animate_world(ds.trajectory, cfg_viz).show()